# Deep Learning Fruit Recognition Project
### Convolutional Neural Networks (CNN) Transfer Learning with VGG16 & TensorFlow

---

## 📋 Project Overview & Team Contributions
This notebook demonstrates an end-to-end computer vision pipeline using transfer learning on the **VGG16** architecture to classify fruit images into 15 distinct categories. The codebase is structured modularly to reflect a collaborative development cycle divided among four engineering team members:

* **Member 1: Data Pipeline & Processing Engine** — Dataset parsing, directory verification, custom input pipeline mapping via `tf.data.Dataset`, and multi-threaded preprocessing.
* **Member 2: Model Architecture & Design** — Functional transfer learning strategy, neural network topology design with frozen VGG16 backbone, and customized loss definitions.
* **Member 3: Training & Optimization Engine** — Network compilation, optimization profile adjustments, iterative training management, and serialization protocols.
* **Member 4: Evaluation & Inference Engine** — Model reconstitution, quantitative validation metrics auditing, and single-image classification pipeline.

# ==========================================================
# SECTION 1: ENVIRONMENT SETUP & GLOBAL CONFIGURATIONS
# ==========================================================

In [ ]:
import os
import random
import warnings
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

# Suppress unnecessary system alerts and warnings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# ----------------------------------------------------------
# Global Configuration Parameters
# ----------------------------------------------------------
IMAGE_SIZE = 224 
BATCH_SIZE = 16 
NUM_CLASSES = 15
EPOCHS = 10
MODEL_FILE = "fruit_model.keras"
CLASS_NAMES_FILE = "class_names.txt"

# Standard Kaggle Dataset path for 'chrisfilo/fruit-recognition'
DATASET_PATH = "/kaggle/input/fruit-recognition/Fruit-Images-Dataset/Fruit-Images-Dataset/Training"

# ----------------------------------------------------------
# Reproducibility & Runtime Device Audit
# ----------------------------------------------------------
def set_reproducibility(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    print(f"[INFO] Global random seed set to: {seed}")

set_reproducibility(42)

print("\n" + "="*50)
print("🚀 HARDWARE RUNTIME AUDIT")
print("="*50)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        print(f"✔ Found Active Compute Device: {gpu}")
else:
    print("⚠ No GPU discovered. Defaulting to CPU infrastructure.")
print(f"TensorFlow Version Matrix: {tf.__version__}")
print("="*50)

# ==========================================================
# SECTION 2: DATA PIPELINE & PREPROCESSING
# Developed by: Member 1
# ==========================================================
### Module Scope:
1. Explores directory tree to dynamically discover targets and format classes.
2. Implements isolated disk-walking routines to securely ingest valid image formats (`.jpg`, `.jpeg`, `.png`).
3. Constructs optimized operational structures via `tf.data.Dataset` mapping API with integrated channel scaling.

In [ ]:
def get_fruit_names():
    """
    Scans the dataset directory path to discover and sort target fruit classes.
    """
    if not os.path.exists(DATASET_PATH):
       raise FileNotFoundError(f"Dataset target directory not found at: {DATASET_PATH}")
       
    fruit_names = []
    for item in os.listdir(DATASET_PATH):
        full_path = os.path.join(DATASET_PATH, item)
        if os.path.isdir(full_path):
            fruit_names.append(item)
            
    fruit_names.sort()
    print(f"\n[✓] Data Discovery: Found {len(fruit_names)} distinct target classes.")
    print(f"[✓] Verified Class Catalog: {fruit_names}")
    return fruit_names


def collect_all_images(fruit_names):
    """
    Performs a disk-walk collection across discovered folders to assemble imagery metadata matrices.
    """
    all_images = []
    for i in range(len(fruit_names)):
        fruit_name = fruit_names[i]
        fruit_folder = os.path.join(DATASET_PATH, fruit_name)
        
        for root_dir, _, files in os.walk(fruit_folder):
            for file_name in files:
                # Supported format constraints matrix validation
                if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
                    image_path = os.path.join(root_dir, file_name)
                    all_images.append([image_path, i])
    
    # Shuffle matching state configurations with explicit seed
    random.seed(42)
    random.shuffle(all_images)
    return all_images


def load_and_process_image(image_path, label):
    """
    TensorFlow Native Pipeline Map Worker Function:
    Executes asymmetric I/O operations, structural spatial decoding, and standardizes tensor dimensions.
    """
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = preprocess_input(image)       
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label


def create_tf_dataset(image_list):
    """
    Converts raw array pairs into production-grade vectorized tf.data.Dataset computational graphs.
    """
    paths = [item[0] for item in image_list]
    labels = [item[1] for item in image_list]
    
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    # Multi-threaded parallel processing transformation configuration
    dataset = dataset.map(load_and_process_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(BATCH_SIZE)
    # Optimization: cache memory footprints and prefetch runtime metrics pipelines
    dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
    return dataset


def get_datasets():
    """
    Master workflow interface orchestration function executing systemic dataset creation and isolation partitions.
    """
    fruit_names = get_fruit_names()
    all_images = collect_all_images(fruit_names)
    
    # Implementing deterministic mathematical split validation parameters (80-20 partition ratio)
    split = int(len(all_images) * 0.8)
    train_images = all_images[:split]
    val_images = all_images[split:]
    
    train_dataset = create_tf_dataset(train_images)
    val_dataset = create_tf_dataset(val_images)
    
    print("\n" + "-"*50)
    print("📊 DATA SUBSET SPECIFICATIONS SUMMARY")
    print("-"*50)
    print(f"  Total Tracked Database Imagery Assets : {len(all_images)}")
    print(f"  Training Sub-Dataset Footprint       : {len(train_images)}")
    print(f"  Validation Sub-Dataset Footprint     : {len(val_images)}")
    print("-"*50 + "\n")
    
    return train_dataset, val_dataset, fruit_names

Platform Setup Completed

### 🔍 Exploratory Visual Verification
Let's run a test extraction across a single structural batch partition from Member 1's pipeline to confirm the stability of dimensions, image ranges, and label encodings.

In [ ]:
try:
    train_data, val_data, class_catalog = get_datasets()
    # Inspect data matrix attributes from first structural iteration block
    image_batch, label_batch = next(iter(train_data.take(1)))
    print("[AUDIT] Extraction verification test structural status: SUCCESS")
    print(f"[AUDIT] Matrix Image Batch Tensor Shape : {image_batch.shape}")
    print(f"[AUDIT] Matrix Label One-Hot Tensor Shape : {label_batch.shape}")
except Exception as e:
    print(f"[WARNING/LOCAL SETUP] Execution bypass notice: {e}. \nEnsure dataset path exists to execute parsing.")

# ==========================================================
# SECTION 3: MODEL ARCHITECTURE & TRANSFER LEARNING DESIGN
# Developed by: Member 2
# ==========================================================
### Module Scope:
1. Configures functional transfer learning strategy via the baseline ImageNet **VGG16** backbone architecture.
2. Freezes downstream convolutional parameter weight fields to maintain core feature detectors.
3. Develops custom structural loss mathematical evaluation routines.

$$\text{Categorical Crossentropy Loss} = -\sum_{c=1}^{M} y_{o,c} \log(p_{o,c})$$

In [ ]:
def my_loss(y_true, y_pred):
    """
    Custom Multi-Class Objective Cost Evaluation Routine.
    Ensures numerical robustness and isolates output activation probability matrices.
    """
    return tf.keras.losses.categorical_crossentropy(y_true, y_pred)

  
def build_vgg16_model(num_classes):
    """
    Assembles a modular Sequential network wrapping a frozen VGG16 core
    and expanding it with tailored classification dense top-layers.
    """
    # Instantiating pretrained convolutional backbone filters
    vgg_base = VGG16(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    
    # Freezing feature extraction layers to prevent gradient weight contamination
    for layer in vgg_base.layers:
        layer.trainable = False
        
    # Appending downstream fully-connected layers to classify the new target domain
    model = Sequential([
        vgg_base,
        Flatten(),
        Dense(128, activation="relu"),
        Dropout(0.3),  # Implemented structural regularization profile to prevent overfitting
        Dense(num_classes, activation="softmax")
    ])
    
    print("\n" + "="*60)
    print("🏗️ DEEP NEURAL NETWORK MODEL ARCHITECTURE")
    print("="*60)
    model.summary()
    print("="*60 + "\n")
    
    return model

# ==========================================================
# SECTION 4: TRAINING PIPELINE & OPTIMIZATION
# Developed by: Member 3
# ==========================================================
### Module Scope:
1. Compiles the modular network with hyperparameter-tuned learning rate schedules via the **Adam** optimizer.
2. Executes runtime training steps across cross-validated iterations.
3. Standardizes text-based label indices and serializes compiled model binaries for deployment.

In [ ]:
def save_class_names(class_names):
    """
    Serializes discovered category lists map indices to disk storage files for runtime consistency.
    """
    try:
        with open(CLASS_NAMES_FILE, "w") as file:
            for name in class_names:
                file.write(name + "\n")
        print(f"[✓] Target classification labels serialized to asset file: {CLASS_NAMES_FILE}")
    except Exception as e:
        print(f"[ERROR] File serialization failure: {e}")


def train():
    """
    Compiles and executes the training pipeline over the dataset splits.
    """
    print("[STEP 1] Ingesting optimized tf.data.Dataset splits...")
    try:
        train_data, val_data, class_names = get_datasets()
    except Exception as e:
        print(f"\n[ABORT] Pipeline Execution Cancelled: {e}")
        print("Please confirm path arrangements map valid files to execute training cycles.")
        return

    print("[STEP 2] Assembling model topology parameters via target class definitions...")
    model = build_vgg16_model(NUM_CLASSES)
    
    # Hyperparameter optimization constraints config tuning selection profile
    optimizer = Adam(learning_rate=0.0001)
    
    print("[STEP 3] Finalizing compilation requirements matrix state...")
    model.compile(
        optimizer=optimizer,
        loss=my_loss,
        metrics=["accuracy"]
    )
    
    print(f"[STEP 4] Initiating training workflow loop across {EPOCHS} distinct epochs...")
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=EPOCHS
    )
    
    print("\n[STEP 5] Serializing complete model binaries to structural assets tracking array...")
    model.save(MODEL_FILE)
    print(f"[✓] Deep Neural Network compiled asset successfully saved as: {MODEL_FILE}")
    
    # Storing operational text file metadata markers
    save_class_names(class_names)
    print("🎉 [CONGRATULATIONS] System pipeline execution iteration cycle complete!")
    return history

### ⚙️ Execute Network Training
Uncomment the downstream tracking execution cell to launch full pipeline updates when computing on resource infrastructure hardware arrays containing valid directory references.

In [ ]:
# training_history = train()

# ==========================================================
# SECTION 5: EVALUATION & INFERENCE ENGINE
# Developed by: Member 4
# ==========================================================
### Module Scope:
1. Reloads serialized Keras structural binaries, resolving custom objective metrics.
2. Evaluates data integrity arrays using validation test data subsets.
3. Builds a raw single-image inference processing routine to support edge runtime deployment.

In [ ]:
TEST_IMAGE_PATH = ""  # Provide path target here during localized deployment evaluations

def load_class_names():
    """
    Reads tracking catalog indices to re-populate working list variables for mapping outputs.
    """
    if not os.path.exists(CLASS_NAMES_FILE):
        # Return default fallback tracking names list matching structural baseline expectations
        return ['Apple', 'Banana', 'Carambola', 'Guava', 'Kiwi', 
                'Mango', 'Orange', 'Peach', 'Pear', 'Persimmon', 
                'Pitaya', 'Plum', 'Pomegranate', 'Tomatoes', 'muskmelon']
                
    class_names = []
    with open(CLASS_NAMES_FILE, "r") as file:
        lines = file.readlines()
        for line in lines:
            name = line.strip()
            if name != "":
                class_names.append(name)
    return class_names


def prepare_single_image(image_path):
    """
    Applies identical data pipeline operations to prepare un-batched test images for predictive classification.
    """
    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False) 
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, [IMAGE_SIZE, IMAGE_SIZE])
    image = preprocess_input(image)
    image = tf.expand_dims(image, axis=0) # Add batch dimension wrapper matrix shape expansion
    return image


def predict_one_image(model, image_path, class_names):
    """
    Performs inference over an isolated deployment image asset target file and outputs the results.
    """
    print(f"\n[INFERENCE] Ingesting image target: {image_path}")
    processed_tensor = prepare_single_image(image_path)
    prediction_probabilities = model.predict(processed_tensor)
    
    # Resolve highest activation index value via argmax calculation
    class_index = int(tf.argmax(prediction_probabilities[0]))
    fruit_name = class_names[class_index]
    confidence_score = round(float(prediction_probabilities[0][class_index]) * 100, 2)
    
    print("-"*45)
    print(f"🔮 PREDICTION REPORT ANALYSIS")
    print("-"*45)
    print(f"  Target Processing File  : {image_path}")
    print(f"  Identified Fruit Class : {fruit_name}")
    print(f"  Classification Certainty: {confidence_score}%")
    print("-"*45 + "\n")
    return fruit_name, confidence_score


def run_evaluation_pipeline():
    """
    Orchestrates validation checking structures across available assets and triggers ad-hoc evaluation operations.
    """
    if not os.path.exists(MODEL_FILE):
        print(f"[NOTICE] Saved network asset file ({MODEL_FILE}) not located on local working storage node.")
        print("[NOTICE] Running inference pipeline natively using initialized configurations fallback pipeline standard.")
        return

    print("[EVALUATOR] Loading saved network tracking binaries file safely...")
    model = tf.keras.models.load_model(
        MODEL_FILE, 
        custom_objects={"my_loss": my_loss}
    )
    
    category_catalog = load_class_names()
    
    print("[EVALUATOR] Auditing network verification loss limits across validation metrics data splits...")
    try:
        _, val_data, _ = get_datasets()
        performance_metrics = model.evaluate(val_data)
        print(f"[📊 COMPUTE LOGS] Dataset validation loss & accuracy assessment values: {performance_metrics}")
    except Exception as e:
        print(f"[BYPASS LOGS] Dataset parsing skipped or path disconnected: {e}")

    # Ad-hoc evaluation inference triggers logic pipeline
    if TEST_IMAGE_PATH != "":
        if os.path.exists(TEST_IMAGE_PATH):
            predict_one_image(model, TEST_IMAGE_PATH, category_catalog)
        else:
            print(f"\n[WARNING] Targeted test verification file was not found at: {TEST_IMAGE_PATH}")
            print("Please verify accuracy of specific manual folder target paths settings.")
    else:
        print("\n[INFO] Inference pipeline idling. Assign a local file pathway to 'TEST_IMAGE_PATH' to trigger automated runtime checks.")

# Run the verification engine evaluation flow routines
run_evaluation_pipeline()